# PDF知识库整理

先查看已有运行案例，再更换PDF执行任务。以下历史记录来自实际本地会话的节选，不会重新调用模型。

In [1]:
from pathlib import Path
import json
from IPython.display import Markdown, display

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "knowledge_maintenance.py").is_file():
    raise RuntimeError("请从仓库根目录打开Notebook。")
records = json.loads((PROJECT_DIR / "examples/runs.json").read_text(encoding="utf-8"))
rows = ["|案例|动作|来源页数|事实记录数|保存拒绝次数|", "|---|---|---|---|---|"]
for r in records["runs"]:
    rows.append(f"|{r['case']}|{r['status']}|{r['source_pages']}|{r['fact_count']}|{r['save_rejections']}|")
display(Markdown("\n".join(rows)))

|案例|动作|来源页数|事实记录数|保存拒绝次数|
|---|---|---|---|---|
|管材单页摘录|new|1|18|0|
|同源完整管材证书|update|4|27|1|
|另一份WCB材料证书|new|2|19|0|
|NST 3N新焊丝证书|new|1|13|0|

## 一次更新与一次新建

完整管材证书补齐摘录时保留旧页ID和18条事实，增加9条。第一次保存因未读取关联页被拒绝，补读后通过。NST 3N证书与旧2N证书身份不同，选择新建。下面查看已有判断与工具顺序。

In [2]:
for r in records["runs"]:
    print(r["case"], "|", r["status"])
    print("工具：", " → ".join(r["tool_path"]))
    print("判断：", r["decision_reason"])
    for e in r["trace_excerpt"]:
        if e.get("event") == "maintenance_rejected":
            print("保存被拒绝：", e["errors"])
    print()
display(Markdown((PROJECT_DIR / "examples/知识页节选.md").read_text(encoding="utf-8")))

管材单页摘录 | new
工具： read_pdf → search_knowledge → search_knowledge → search_knowledge → search_knowledge → read_library_page → commit_page
判断： 检索未发现同一出具方或同一证书编号的旧页；唯一候选为 NST 焊丝证书（10972205，炉号 114000），出具方、编号、产品与炉号均不同，故新建独立页，仅作主题相关关联。

同源完整管材证书 | update
工具： read_pdf → search_knowledge → search_knowledge → read_library_page → commit_page → read_library_page → commit_page
判断： 本次PDF出具方、证书号4-12-12567-rev.01、炉号QM414、质量批号QL40012275、SMST-Item与Part No均与旧页K-520b83cd20531556一致，本次为完整4页，补齐旧页缺失的第2–4页且无冲突，故更新原页。
保存被拒绝： ['关联旧页前需读取该页正文。']

另一份WCB材料证书 | new
工具： read_pdf → search_knowledge → search_knowledge → search_knowledge → search_knowledge → read_library_page → read_library_page → commit_page
判断： 检索未见证书号 20150410002-1/-2、炉号 HM305/HM105 或订单 HM-GXN-T-030615 的旧页；已读的两个候选页出具方、证书号与对象均不同，仅同属 EN 10204 3.1 材料证书主题，故新建独立页并关联。

NST 3N新焊丝证书 | new
工具： read_pdf → search_knowledge → search_knowledge → read_library_page → search_knowledge → commit_page
判断： 本次证书号 1001378235、产品 NST Carbomig 3N、炉号 878730 与库中 NST Carbomig 2

# NST 3N知识页节选

以下两段来自已保存的模型正文，仅选取证书信息和与旧页的关系；未改写段内内容。完整知识页在本地生成。

## 二、证书基本信息 [PDF第1页]

| 项目 | 内容 |
|---|---|
| 证书号 Cert no | 1001378235 |
| Date | 18-01-2023 |
| Manufacturing Date | 16-01-2023 |
| PURCHASE ORDER | 12445 / 06.12.22 |
| 产品名称 PRODUCT NAME | NST Carbomig 3N |
| 产品类型 TYPE OF PRODUCT | Solid MAG Wire |
| 炉号 HEAT N° | 878730 |

原文日期写作 "18-01-2023"、"16-01-2023"，采购订单写作 "12445 / 06.12.22"；原文未说明日期中日/月的书写约定，此处按原文保留。

## 九、与知识库中相关页面的关系

- 与 NST Carbomig 2N 证书 10972205（炉号 114000）为同一出具方、同一系列实心 MAG 焊丝，但产品型号、证书号、炉号、日期、化学成分与力学数值均不同，属独立资料，未合并、未覆盖；两页数据不可互相替代或混用。

[公开原PDF](https://nst.no/dokumenter/sertifikater/878730%2008%2015kg.pdf)


## 更换PDF实际运行

下面的单元会调用模型并生成本地知识库与对照HTML。先配置 `.env`，将PDF放到 `samples/input.pdf` 或修改路径，再运行。仅查看上面的记录无需执行此部分。

In [ ]:
PDF_PATH = PROJECT_DIR / "samples/input.pdf"
TASK = "整理主要信息、条件、数值和表格；查找已有资料，判断新建、更新或待确认，并保留页码依据。"
LIBRARY_DIR = PROJECT_DIR / "knowledge_base_v2"

from presentation import process_pdf, show_presentation

SESSION = None
SESSION = process_pdf(str(PDF_PATH), TASK, LIBRARY_DIR)
show_presentation(SESSION)
print("本地展示页面：", SESSION["latest_html"])